# Specific Stock Simulation

Get the historical daily data for one ticker, run the prediction analysis across the full history, simulate trades from the resulting daily signals, and show a consolidated chart with price, trade markers, and portfolio value.

Trading rules:
- `STRONG BUY`: buy stock worth 10% of `initial_funds`
- `WEAK BUY`: buy stock worth 5% of `initial_funds`
- `HOLD`: do nothing
- `WEAK SELL`: sell 5% of current holdings
- `STRONG SELL`: sell 10% of current holdings

Fractional shares are allowed.


In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import analysis_blocks


## Block 1: Parameters


In [2]:
ticker = "UNH"
initial_funds = 10000
include_fundamentals = True
include_sentiment = False  # True requires API keys and adds latency


In [3]:

df_pred = analysis_blocks.build_prediction_frame(
            ticker,
            include_sentiment=include_sentiment
        )
df_pred

(           Date        Open        High         Low       Close   Adj Close  \
 0    2026-03-13  278.059998  282.880005  277.809998  282.089996  282.089996   
 1    2026-03-12  284.779999  287.559998  276.290009  277.049988  277.049988   
 2    2026-03-11  282.380005  286.220001  280.600006  285.250000  285.250000   
 3    2026-03-10  287.380005  287.579987  278.899994  282.339996  282.339996   
 4    2026-03-09  282.000000  285.200012  278.660004  285.170013  285.170013   
 ...         ...         ...         ...         ...         ...         ...   
 1251 2021-03-19  361.920013  370.459991  359.010010  365.579987  335.557220   
 1252 2021-03-18  352.869995  364.320007  352.339996  362.049988  332.317108   
 1253 2021-03-17  355.170013  358.070007  351.829987  352.179993  323.257660   
 1254 2021-03-16  353.529999  355.359985  351.910004  354.600006  325.478973   
 1255 2021-03-15  356.920013  357.940002  351.549988  353.880005  324.818085   
 
         Volume TICKER       SMA10    

## Block 4: Simulate Daily Buy/Sell/Hold Strategy


In [4]:
simulation_result = analysis_blocks.simulate_prediction_signal_strategy(
    df_pred,
    initial_funds=initial_funds,
)


TypeError: tuple indices must be integers or slices, not list

## Block 5: Portfolio Summary


In [ ]:
simulation_result["price_history"]

In [ ]:
simulation_result["daily_history"]

In [ ]:
simulation_result["transactions"]


In [ ]:
simulation_result["summary"]


## Block 6: Consolidated Chart


In [ ]:
price_history = simulation_result["price_history"].copy()
daily_history = simulation_result["daily_history"].copy()
transactions = simulation_result["transactions"].copy()
simulation_summary = simulation_result["summary"].iloc[0]

buy_txns = transactions[transactions["action"] == "BUY"].copy()
sell_txns = transactions[transactions["action"] == "SELL"].copy()

fig, ax_price = plt.subplots(figsize=(16, 8))
ax_price.plot(
    price_history["Date"],
    price_history["Close"],
    color="#1f77b4",
    linewidth=2,
    label=f"{ticker} close",
)

if not buy_txns.empty:
    ax_price.scatter(
        buy_txns["Date"],
        buy_txns["trade_price"],
        color="#2ca02c",
        marker="^",
        s=90,
        label="Buy trades",
        zorder=5,
    )

if not sell_txns.empty:
    ax_price.scatter(
        sell_txns["Date"],
        sell_txns["trade_price"],
        color="#d62728",
        marker="v",
        s=90,
        label="Sell trades",
        zorder=5,
    )

ax_price.set_xlabel("Date")
ax_price.set_ylabel("Share price ($)")
ax_price.grid(alpha=0.25)

ax_portfolio = ax_price.twinx()
ax_portfolio.plot(
    daily_history["Date"],
    daily_history["portfolio_value"],
    color="#ff7f0e",
    linewidth=2,
    label="Portfolio value",
)
ax_portfolio.set_ylabel("Portfolio value ($)")

title = (
    f"{ticker} daily signal simulation | "
    f"Units held: {simulation_summary['units_held']:.4f} | "
    f"P/L: {simulation_summary['profit_loss_pct']:.2f}%"
)
ax_price.set_title(title)

handles_price, labels_price = ax_price.get_legend_handles_labels()
handles_portfolio, labels_portfolio = ax_portfolio.get_legend_handles_labels()
ax_price.legend(handles_price + handles_portfolio, labels_price + labels_portfolio, loc="upper left")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Block 7: Daily Portfolio History


In [ ]:
simulation_result["daily_history"][[
    "Date",
    "signal_text",
    "action",
    "trade_units",
    "trade_value",
    "cash_balance",
    "units_held",
    "holdings_value",
    "portfolio_value",
    "profit_loss_pct",
]].tail(60)
